**Code:** [GitHub · Kukai-7/industrial-data-valuation](https://github.com/Kukai-7/industrial-data-valuation) · **Data:** [Hugging Face · synthetic-industrial-data-valuation](https://huggingface.co/datasets/Kukai-7/synthetic-industrial-data-valuation) · **Demo:** [Space](https://huggingface.co/spaces/Kukai-7/industrial-data-valuation-demo)

**Abstract.** This notebook generates three synthetic industrial tables (sensors, maintenance, supply chain) linked by consistent keys, to illustrate valuation methods for data as an intangible asset. Limits first: no real data, plausible but invented parameters, and a closing illustration that gives a relative ranking, not a price.

*Résumé (FR). Trois tables industrielles synthétiques reliées par des clés cohérentes, pour illustrer la valorisation de la donnée comme actif incorporel. Aucune donnée réelle ; classement relatif, pas un prix.*

# Synthetic data generation

> *Synthetic data generated for methodological illustration. No real industrial data.*

Three tables, one fixed seed, a Parquet export to `data/synthetic/`.

| Table | Grain | Keys |
|---|---|---|
| `sensors` | one reading every 6 hours, 40 assets × 3 measures, 180 days | `reading_id`, `asset_id`, `sensor_id` |
| `maintenance` | one work order | `work_order_id`, `asset_id` → `sensors`, `purchase_order_id` → `supply_chain` |
| `supply_chain` | one purchase order for parts | `purchase_order_id`, `supplier_id` |

The links between tables are deliberate: an asset's vibration rises in the three days before a failure, every part used comes from a purchase order of the same family already delivered, and each supplier has its own defect rate.

In [ ]:
# Synthetic data generated for methodological illustration. No real industrial data.
import numpy as np
import pandas as pd

SEED = 2026
rng = np.random.default_rng(SEED)

N_ASSETS, N_SUPPLIERS = 40, 12
START = pd.Timestamp("2025-01-01")
DAYS = 180
ASSETS = [f"AS-{i:03d}" for i in range(1, N_ASSETS + 1)]
SUPPLIERS = [f"SUP-{i:02d}" for i in range(1, N_SUPPLIERS + 1)]
FAMILIES = ["bearing", "seal", "motor", "valve", "sensor", "controller"]

## 1 · Supply chain

In [ ]:
# ---------------------------------------------------------------- supply_chain: purchase orders for parts
defect_base = rng.beta(2, 18, N_SUPPLIERS)                # defect rate specific to each supplier
lead_base = rng.integers(7, 46, N_SUPPLIERS)              # promised lead time specific to each supplier (days)
base_price = dict(zip(FAMILIES, [180, 45, 1200, 320, 260, 900]))
n_po = 900
sup = rng.integers(0, N_SUPPLIERS, n_po)
fam = rng.choice(FAMILIES, n_po)
order = START + pd.to_timedelta(rng.integers(0, DAYS, n_po), unit="D")
promised = order + pd.to_timedelta(lead_base[sup], unit="D")
delay = np.maximum(0, rng.normal(2, 6, n_po)).round()
supply_chain = pd.DataFrame({
    "purchase_order_id": [f"PO-{i:05d}" for i in range(1, n_po + 1)],
    "supplier_id": np.array(SUPPLIERS)[sup],
    "part_family": fam,
    "order_date": order,
    "promised_date": promised,
    "delivered_date": promised + pd.to_timedelta(delay, unit="D"),
    "quantity": rng.integers(1, 40, n_po),
    "unit_price_eur": np.round([base_price[f] * rng.lognormal(0, .18) for f in fam], 2),
    "defect_rate": np.round(np.clip(defect_base[sup] + rng.normal(0, .01, n_po), 0, 1), 4),
})
supply_chain["lead_time_days"] = (supply_chain.delivered_date - supply_chain.order_date).dt.days

## 2 · Maintenance

In [ ]:
# ---------------------------------------------------------------- maintenance: work orders
n_wo = 1200
asset = rng.integers(0, N_ASSETS, n_wo)
wtype = rng.choice(["preventive", "corrective", "inspection"], n_wo, p=[.45, .35, .20])
opened = START + pd.to_timedelta(rng.integers(0, DAYS * 24, n_wo), unit="h")
downtime = np.where(wtype == "corrective", rng.gamma(2.2, 4.0, n_wo), np.where(wtype == "preventive", rng.gamma(1.5, 1.5, n_wo), rng.gamma(1.0, .5, n_wo)))
fam_wo = rng.choice(FAMILIES, n_wo)
uses_part = (wtype != "inspection") & (rng.random(n_wo) < .7)
by_family = {f: np.flatnonzero(fam == f) for f in FAMILIES}      # the part ordered belongs to the same family as the component
delivered = (promised + pd.to_timedelta(delay, unit="D")).values
def pick_po(f, t):                                             # a part already delivered when the order opens; otherwise no part
    c = by_family[f]; ok = c[delivered[c] <= t.to_datetime64()]
    return rng.choice(ok) if len(ok) else -1
po_idx = np.array([pick_po(f, t) for f, t in zip(fam_wo, opened)])
uses_part = uses_part & (po_idx >= 0)
po_idx = np.maximum(po_idx, 0)
maintenance = pd.DataFrame({
    "work_order_id": [f"WO-{i:05d}" for i in range(1, n_wo + 1)],
    "asset_id": np.array(ASSETS)[asset],
    "opened_at": opened,
    "closed_at": opened + pd.to_timedelta(np.round(downtime * 60), unit="min"),
    "work_type": wtype,
    "component_family": fam_wo,
    "downtime_hours": np.round(downtime, 2),
    "labour_hours": np.round(downtime * rng.uniform(.6, 1.4, n_wo) + 1, 2),
    "purchase_order_id": np.where(uses_part, supply_chain.purchase_order_id.values[po_idx], None),
    "supplier_id": np.where(uses_part, supply_chain.supplier_id.values[po_idx], None),
})
maintenance["parts_cost_eur"] = np.where(uses_part, np.round(supply_chain.unit_price_eur.values[po_idx] * rng.integers(1, 5, n_wo), 2), 0.0)

## 3 · Sensors

In [ ]:
# ---------------------------------------------------------------- sensors: one reading every 6 hours
steps = pd.date_range(START, periods=DAYS * 4, freq="6h")
METRICS = {"temperature_c": (65, 4), "vibration_mm_s": (2.8, .5), "power_kw": (45, 6)}
corr = maintenance[maintenance.work_type == "corrective"]
blocks = []
for a_i, a in enumerate(ASSETS):
    drift = np.zeros(len(steps))
    for t in corr.opened_at[corr.asset_id == a]:            # vibration rises in the 3 days before a failure
        k = np.searchsorted(steps, t)
        drift[max(0, k - 12):k] += np.linspace(0, .9, min(12, k))
    level = rng.normal(0, .05, 3)
    for m_i, (m, (mu, sd)) in enumerate(METRICS.items()):
        v = mu * (1 + level[m_i]) + rng.normal(0, sd, len(steps))
        if m == "vibration_mm_s": v = v + drift
        missing = rng.random(len(steps)) < .02              # 2% of readings missing
        blocks.append(pd.DataFrame({"asset_id": a, "sensor_id": f"{a}-{m_i + 1}", "timestamp": steps, "metric": m,
                                    "value": np.where(missing, np.nan, np.round(v, 3)), "quality_flag": np.where(missing, "missing", "ok")}))
sensors = pd.concat(blocks, ignore_index=True)
sensors.insert(0, "reading_id", np.arange(1, len(sensors) + 1))

TABLES = {"sensors": sensors, "maintenance": maintenance, "supply_chain": supply_chain}
for t in TABLES.values():                                     # explicit time unit (stable across pandas versions)
    for c in t.columns:
        if str(t[c].dtype).startswith("datetime64"): t[c] = t[c].astype("datetime64[ms]")

## 4 · Consistency checks

In [ ]:
m = maintenance.dropna(subset=["purchase_order_id"]).merge(supply_chain, on="purchase_order_id", suffixes=("", "_po"))
checks = {
    "rows per table": {k: len(v) for k, v in TABLES.items()},
    "unique primary keys": all(TABLES[t][k].is_unique for t, k in [("sensors", "reading_id"), ("maintenance", "work_order_id"), ("supply_chain", "purchase_order_id")]),
    "maintenance assets present in sensors": set(maintenance.asset_id) <= set(sensors.asset_id),
    "supplier matches the purchase order": bool((m.supplier_id == m.supplier_id_po).all()),
    "part from the same family as the component": bool((m.component_family == m.part_family).all()),
    "part delivered before the work order opens": bool((m.delivered_date <= m.opened_at).all()),
    "share of missing readings": round(float(sensors.value.isna().mean()), 4),
}
checks

## 5 · Parquet export

In [ ]:
import os
os.makedirs("../data/synthetic", exist_ok=True)
for name, t in TABLES.items():
    t.to_parquet(f"../data/synthetic/{name}.parquet", index=False)
    print(f"data/synthetic/{name}.parquet", t.shape)

## 6 · Illustration: the three approaches on these tables

**Limits before results.** Every economic assumption below (rebuild costs, reference prices, cost of an hour of downtime) is **invented for the example**. The result is a relative ranking of the three tables, not an opposable price.

- **Cost:** what it would take to rebuild the table (sensors installed, share of a software tool, entry and cleaning time).
- **Market:** volume × reference price per record, minus a 50% discount for the lack of comparables.
- **Income:** yearly gains the table makes possible, discounted over 5 years at 12%; they are measured on the data itself (failures preceded by a vibration alert, defect gap between suppliers).

The three amounts are blended by confidence (as in notebook 01) to give $Val$, then fed into the reference formula $V = QRU \times \sum (Val \times ICF) / V_p$: here $QRU$ is the completeness of the table, $ICF = 1$ (one-off transfer), $V_p = 1.05$ (light processing), and the sum runs over the three tables of the portfolio.

In [ ]:
H = dict(infra={"sensors": 180_000, "maintenance": 30_000, "supply_chain": 20_000},   # €: sensors installed, share of a CMMS / ERP tool (assumption)
         hours_per_record={"sensors": 0, "maintenance": .25, "supply_chain": .2},        # hours of entry and cleaning per record (assumption)
         cost_per_hour=45, ref_price={"sensors": .01, "maintenance": 2.0, "supply_chain": 1.0},  # €/hour, €/record (assumptions)
         discount=.5, downtime_cost_per_hour=500, capture=.3, horizon=5, rate=.12,
         weights={"cost": .30, "market": .25, "income": .45}, confidence={"cost": .9, "market": .5, "income": .7},
         icf=1.0, vp=1.05)
annuity = (1 - (1 + H["rate"]) ** -H["horizon"]) / H["rate"]
per_year = 365 / 180                                                                   # annualise the 180-day window

# Sensors: share of failures preceded by abnormal vibration (threshold: mean + 2 standard deviations) and avoidable downtime
vib = sensors[sensors.metric == "vibration_mm_s"].set_index(["asset_id", "timestamp"]).value.sort_index()
threshold = vib.mean() + 2 * vib.std()
failures = maintenance[maintenance.work_type == "corrective"]
detected = [bool((vib.loc[a].loc[t - pd.Timedelta("3D"):t] > threshold).any()) for a, t in zip(failures.asset_id, failures.opened_at)]
gain_sensors = failures.downtime_hours[detected].sum() * H["downtime_cost_per_hour"] * H["capture"] * per_year
# Maintenance: planning part of the failures brings their duration down to that of a preventive job
gap = failures.downtime_hours.mean() - maintenance[maintenance.work_type == "preventive"].downtime_hours.mean()
gain_maint = gap * len(failures) * H["downtime_cost_per_hour"] * H["capture"] * .5 * per_year
# Supply chain: buying from the third of suppliers with the fewest defects
spend = (supply_chain.quantity * supply_chain.unit_price_eur).sum()
avg = supply_chain.groupby("supplier_id").defect_rate.mean()
gain_supply = spend * (supply_chain.defect_rate.mean() - avg.nsmallest(4).mean()) * per_year

gains = {"sensors": gain_sensors, "maintenance": gain_maint, "supply_chain": gain_supply}
w = {k: H["weights"][k] * H["confidence"][k] for k in H["weights"]}
rows = []
for name, t in TABLES.items():
    a = dict(cost=H["infra"][name] + len(t) * H["hours_per_record"][name] * H["cost_per_hour"],
             market=len(t) * H["ref_price"][name] * (1 - H["discount"]),
             income=gains[name] * annuity)
    val = sum(w[k] * a[k] for k in a) / sum(w.values())                              # Val: confidence-weighted blend of the three approaches
    qru = 1 - t.isna().any(axis=1).mean() if name == "sensors" else 1.0            # QRU: here the completeness of the records
    rows.append(dict(table=name, **a, Val=val, QRU=qru, V=qru * val * H["icf"] / H["vp"]))
V = pd.DataFrame(rows).set_index("table")
V["rank"] = V.V.rank(ascending=False).astype(int)
print(f"Failures preceded by a vibration alert: {np.mean(detected):.0%} (threshold {threshold:.2f} mm/s)")
print(f"Portfolio (sum of the three tables): {V.V.sum():,.0f} relative units")
V.round(2)

**Reading.** The income leg, measured on the data itself, drives most of the gap between tables, as in notebook 01. Changing a single invented assumption (the cost of an hour of downtime, for example) can reverse the ranking: that is why notebook 01 runs a sensitivity analysis.

## Reproducibility & License

- **Running:** *Run All* is enough; seed `SEED = 2026`, no external data.
- **Data:** *Synthetic data generated for methodological illustration. No real industrial data.*
- **License:** code under the MIT License; data under CC BY 4.0.